In [3]:
import ollama
import requests
from bs4 import BeautifulSoup
from IPython.display import display, Markdown

In [4]:
def fetch_website_contents(url):
    """
    Fetch and extract text content from a website
    """
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'html.parser')

        # Remove script and style elements
        for script in soup(["script", "style"]):
            script.decompose()

        # Get text and clean it
        text = soup.get_text()
        lines = (line.strip() for line in text.splitlines())
        chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
        text = ' '.join(chunk for chunk in chunks if chunk)

        # Limit text to avoid token limits
        if len(text) > 8000:
            text = text[:8000] + "..."

        return text
    except Exception as e:
        return f"Error fetching website: {str(e)}"

In [5]:
def messages_for(website):
    """
    Prepare the messages for the LLM
    """
    # For Ollama, we need to format differently than OpenAI
    return f"""
Please provide a concise summary of the following website content.
Focus on the main topics, key information, and the overall purpose.

Website Content:
{website}

Summary:"""

In [7]:
def summarize(url):
    website = fetch_website_contents(url)

    # Using Ollama
    response = ollama.chat(
        model="llama3:latest",
        messages=[
            {
                'role': 'system',
                'content': 'You are a helpful assistant that provides clear, concise summaries of website content.'
            },
            {
                'role': 'user',
                'content': messages_for(website)
            }
        ],
        options={
            'temperature': 0.7,
            'num_predict': 300  # Limit response length
        }
    )

    return response['message']['content']

# Test the function
summarize("https://edwarddonner.com")

'Here is a concise summary of the website content:\n\nThe website belongs to Edward Donner, co-founder and CTO of AI startup Nebula.io. He is also a bestselling Udemy course creator with over 600,000 enrolled students from 194 countries. The main topics on his website are:\n\n* Curriculum: A collection of courses on LLMs (Large Language Models) and AI engineering, including "Avatar" which pits LLMs against each other in a battle of diplomacy and deviousness.\n* Proficiency: Resources for learning and improving skills in AI coding, building, and engineering.\n* C4Outsmart: An arena where LLMs compete against each other.\n\nThe website\'s purpose is to provide a platform for Ed to share his knowledge and expertise in AI and LLMs with others. He invites visitors to learn from him and engage with his digital avatar.'

In [8]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

display_summary("https://edwarddonner.com")

Here is a concise summary of the website content:

The website belongs to Edward Donner, a co-founder and CTO of AI startup Nebula.io. He has created best-selling Udemy courses on LLMs (Large Language Models) and is passionate about sharing his knowledge with others. The main topics covered on the website include:

* Curriculum: A collection of courses and resources on AI coding, building, and engineering.
* Proficiency: Information on how to improve skills in AI and related fields.
* C4Outsmart: A simulated arena where LLMs compete against each other.

The overall purpose of the website is to provide a platform for learning and sharing knowledge about AI and LLMs. Edward Donner invites visitors to engage with him and his digital avatar, and offers resources and courses for those interested in pursuing a career in AI engineering.